# InflationCast
## Machine Learning-Based Forecasting of India's Consumer Price Inflation Using Macroeconomic Indicators

**Applied Machine Learning Capstone** | Type: Regression
**Team:** Brij Nandan Rawat (Batch 10), Suvansh Bansal (Batch 10)
**Data source:** Ministry of Statistics & Programme Implementation (MoSPI), Government of India, eSankhyiki portal

**Goal:** predict next month's All-India CPI inflation (YoY %) from past inflation, seasonality and macro indicators.

> **How to use:** put your dataset at `data/raw/master.csv` (columns: `date`, `inflation`, plus macro columns such as `repo_rate`, `usd_inr`, `brent`, `iip_yoy`).
> If that file is missing, the notebook runs on **synthetic demo data** so you can test it. Never report demo results.

## 1. Setup

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 4.5)
SEED = 42
DATA_PATH = "data/raw/master.csv"   # your real dataset
TEST_MONTHS = 24                    # last N months held out for testing

## 2. Load data
`master.csv` needs one row per month: `date` (YYYY-MM), `inflation` (CPI combined YoY %), and any macro columns.
Every extra numeric column is used as a feature automatically.

In [ ]:
def make_demo(n_months=156, seed=SEED):
    """SYNTHETIC data to test the pipeline. NOT real inflation."""
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2013-01-01", periods=n_months, freq="MS")
    brent = 70 + np.cumsum(rng.normal(0, 3, n_months)).clip(-35, 40)
    usd_inr = 62 + np.cumsum(rng.normal(0.15, 0.5, n_months))
    repo = (6.5 + np.cumsum(rng.normal(0, 0.12, n_months))).clip(4, 8)
    iip = rng.normal(4, 3, n_months)
    infl = np.zeros(n_months); infl[0] = 5
    for t in range(1, n_months):
        seas = 0.6 * np.sin(2 * np.pi * (idx[t].month - 3) / 12)
        infl[t] = (0.85*infl[t-1] + 0.02*(brent[t-1]-70) + 0.05*(usd_inr[t-1]-usd_inr[t-2])
                   + 0.3 + 0.3*seas + rng.normal(0, 0.35))
    return pd.DataFrame({"inflation": infl.round(2), "repo_rate": repo.round(2),
                         "usd_inr": usd_inr.round(2), "brent": brent.round(2),
                         "iip_yoy": iip.round(2)}, index=idx.rename("date"))

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    df["date"] = pd.to_datetime(df["date"]).dt.to_period("M").dt.to_timestamp()
    df = df.sort_values("date").drop_duplicates("date").set_index("date")
    DEMO = False
    print(f"Loaded real data: {DATA_PATH}")
else:
    df = make_demo()
    DEMO = True
    print("!! DEMO MODE: synthetic data - results are NOT real inflation results !!")

assert "inflation" in df.columns, "master.csv must contain an 'inflation' column"
print(df.shape, "|", df.index.min().strftime("%b %Y"), "->", df.index.max().strftime("%b %Y"))
df.head()

## 3. Exploratory data analysis

In [ ]:
display(df.describe().round(2))
print("Missing values per column:\n", df.isna().sum())

In [ ]:
ax = df["inflation"].plot(title="CPI inflation (YoY %)", color="tab:red")
ax.set_ylabel("%"); ax.grid(alpha=.3); plt.show()

macro_cols = [c for c in df.columns if c != "inflation"]
if macro_cols:
    fig, axes = plt.subplots(len(macro_cols), 1, figsize=(10, 2.2*len(macro_cols)), sharex=True)
    axes = np.atleast_1d(axes)
    for a, c in zip(axes, macro_cols):
        df[c].plot(ax=a, title=c); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()

In [ ]:
corr = df.corr().round(2)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, corr.iloc[i, j], ha="center", va="center", fontsize=8)
plt.title("Correlation matrix"); plt.colorbar(im); plt.tight_layout(); plt.show()

## 4. Feature engineering
Target = **next month's inflation** (1-step-ahead). Features: current and lagged inflation (1, 2, 3, 6, 12),
rolling means, month-on-month change, seasonality (sin/cos of month), and each macro column's level, lags and change.

In [ ]:
def build_features(df, horizon=1, infl_lags=(1, 2, 3, 6, 12), macro_lags=(1, 3)):
    X = pd.DataFrame(index=df.index)
    infl = df["inflation"]
    X["infl_t"] = infl
    for l in infl_lags:
        X[f"infl_lag{l}"] = infl.shift(l)
    X["infl_roll3"] = infl.rolling(3).mean()
    X["infl_roll6"] = infl.rolling(6).mean()
    X["infl_diff1"] = infl.diff()
    X["month_sin"] = np.sin(2*np.pi*df.index.month/12)
    X["month_cos"] = np.cos(2*np.pi*df.index.month/12)
    for c in [c for c in df.columns if c != "inflation"]:
        X[c] = df[c]
        for l in macro_lags:
            X[f"{c}_lag{l}"] = df[c].shift(l)
        X[f"{c}_chg1"] = df[c].diff()
    if horizon == 0:                       # latest-row features, used for forecasting
        return X.dropna(), None
    y = infl.shift(-horizon).rename("target")
    data = pd.concat([X, y], axis=1).dropna()
    return data.drop(columns="target"), data["target"]

X, y = build_features(df)
print(f"{len(X)} usable months, {X.shape[1]} features")
X.head()

## 5. Train / test split
**Time-based split** - never shuffle a time series, otherwise the model sees the future.

In [ ]:
split = len(X) - TEST_MONTHS
Xtr, Xte, ytr, yte = X.iloc[:split], X.iloc[split:], y.iloc[:split], y.iloc[split:]
print(f"Train: {len(Xtr)} months ({Xtr.index.min():%b %Y} -> {Xtr.index.max():%b %Y})")
print(f"Test : {len(Xte)} months ({Xte.index.min():%b %Y} -> {Xte.index.max():%b %Y})")

## 6. Models
Baseline = **persistence** ("next month = this month"). A model must beat it to be worth reporting.
Cross-validation uses `TimeSeriesSplit` on the training set only.

In [ ]:
def metrics(yt, yp):
    return {"RMSE": np.sqrt(mean_squared_error(yt, yp)),
            "MAE": mean_absolute_error(yt, yp), "R2": r2_score(yt, yp)}

models = {
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "RandomForest": RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=SEED),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=SEED),
}
try:
    from xgboost import XGBRegressor
    models["XGBoost"] = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=SEED)
except ImportError:
    print("xgboost not installed - skipping (pip install xgboost to include it)")

rows, preds, fitted = [], {}, {}
preds["Persistence baseline"] = Xte["infl_t"]
rows.append({"Model": "Persistence baseline", **metrics(yte, Xte["infl_t"]), "CV_RMSE": np.nan})

for name, m in models.items():
    cv = -cross_val_score(m, Xtr, ytr, cv=TimeSeriesSplit(5), scoring="neg_root_mean_squared_error").mean()
    m.fit(Xtr, ytr)
    preds[name] = pd.Series(m.predict(Xte), index=yte.index)
    fitted[name] = m
    rows.append({"Model": name, **metrics(yte, preds[name]), "CV_RMSE": cv})

results = pd.DataFrame(rows).sort_values("RMSE").round(3).reset_index(drop=True)
results

## 7. Evaluation

In [ ]:
best = results[results.Model != "Persistence baseline"].iloc[0]["Model"]
print("Best ML model on the test set:", best)

plt.plot(yte.index, yte, "k-o", ms=3, label="Actual")
plt.plot(yte.index, preds["Persistence baseline"], ":", label="Persistence baseline")
for n in [m for m in preds if m not in ("Persistence baseline",)]:
    plt.plot(yte.index, preds[n], "--", alpha=1 if n == best else .35, lw=2 if n == best else 1, label=n)
plt.title("Actual vs predicted CPI inflation (test period)")
plt.ylabel("YoY inflation (%)"); plt.legend(); plt.grid(alpha=.3); plt.show()

In [ ]:
# Residuals of the best model
res_ = yte - preds[best]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(res_.index, res_, "o-", ms=3); ax[0].axhline(0, color="k", lw=.8)
ax[0].set_title(f"{best} - residuals over time"); ax[0].grid(alpha=.3)
ax[1].hist(res_, bins=10, edgecolor="k"); ax[1].set_title("Residual distribution")
plt.tight_layout(); plt.show()

## 8. Feature importance

In [ ]:
if "RandomForest" in fitted:
    imp = pd.Series(fitted["RandomForest"].feature_importances_, index=X.columns).sort_values()
    imp.tail(12).plot.barh(title="Random Forest - top 12 features", figsize=(7, 5))
    plt.tight_layout(); plt.show()

## 9. Forecast the next month

In [ ]:
final = type(models[best])(**models[best].get_params()) if best != "Ridge" else make_pipeline(StandardScaler(), Ridge(alpha=1.0))
final.fit(X, y)                      # refit on all labelled data
latest, _ = build_features(df, horizon=0)
nxt = float(final.predict(latest.iloc[[-1]])[0])
nxt_month = (latest.index[-1] + pd.offsets.MonthBegin(1)).strftime("%b %Y")
print(f"Model: {best} | Forecast CPI inflation for {nxt_month}: {nxt:.2f}%")
if DEMO: print("(demo data - ignore this number)")

## 10. Save results

In [ ]:
os.makedirs("outputs", exist_ok=True)
results.to_csv("outputs/model_comparison.csv", index=False)
print("Saved outputs/model_comparison.csv")

## 11. (Optional) Merge MoSPI 2026 division / group files
Put the downloaded `Annexures_for_Press_Release_*.xlsx` files in a folder and call `build_mospi_tables(folder)`.
It reads the month from each file's title row and returns one tidy table. Use it for extra analysis on 2026 (2024-base series);
do **not** mix 2024-base division columns with the older 2012-base series.

In [ ]:
import re, glob, openpyxl
_M = {m: i for i, m in enumerate(["january","february","march","april","may","june","july",
      "august","september","october","november","december"], 1)}

def _parse_annex(path):
    ws = openpyxl.load_workbook(path, data_only=True).worksheets[0]
    rows = list(ws.iter_rows(values_only=True))
    title = " ".join(str(c) for r in rows[:3] for c in r if c)
    m = re.search(r"for\s+([A-Za-z]+),?\s+(\d{4})", title)
    date = pd.Timestamp(year=int(m.group(2)), month=_M[m.group(1).lower()], day=1)
    is_group = "group" in title.lower() and "division" not in title.lower()
    out = []
    for r in rows:
        nums = [c for c in r if isinstance(c, (int, float))]
        if len(nums) != 6: continue
        txt = [str(c).strip().replace("\xa0", " ") for c in r if isinstance(c, str) and c.strip()]
        code_, name = (txt[1], txt[2]) if is_group else ((txt[0], txt[1]) if len(txt) >= 2 else ("ALL", txt[0]))
        out.append(dict(date=date, code=code_, name=name,
                        idx_rural=nums[0], idx_urban=nums[1], idx_comb=nums[2],
                        infl_rural=nums[3], infl_urban=nums[4], infl_comb=nums[5]))
    return ("group" if is_group else "division"), pd.DataFrame(out)

def build_mospi_tables(folder):
    div, grp = [], []
    for f in glob.glob(os.path.join(folder, "*.xlsx")):
        kind, d = _parse_annex(f)
        (grp if kind == "group" else div).append(d)
    cat = lambda L: pd.concat(L).drop_duplicates(["date", "code", "name"]).sort_values(["date", "code"]) if L else pd.DataFrame()
    return cat(div), cat(grp)

# Example (uncomment and point to your folder):
# div_tbl, grp_tbl = build_mospi_tables("data/mospi_annexures")
# div_tbl.head()